In [ ]:
import json,glob,time,importlib.util,subprocess,sys,torch
if importlib.util.find_spec('transformers') is None:
 subprocess.check_call([sys.executable,'-m','pip','install','-q','transformers','accelerate'])
from transformers import AutoModelForCausalLM,AutoTokenizer
assert torch.cuda.is_available()
cases=json.load(open(glob.glob('/kaggle/input/**/cases.json',recursive=True)[0]))
model_id='Qwen/Qwen2.5-3B-Instruct'
t=time.time();tokenizer=AutoTokenizer.from_pretrained(model_id);model=AutoModelForCausalLM.from_pretrained(model_id,torch_dtype=torch.float16,device_map='auto');load=time.time()-t
rules={
'A':'Suggest one concrete research topic and explain why it fits the student and lab. Answer in Korean.',
'B':'Suggest one feasible research exploration for the student. Distinguish (1) research question, (2) what the paper actually states, (3) a specific course concept and how it might help, and (4) unknowns and a small first step. Cite the exact paper ID and course ID when used. Use only supplied evidence for factual claims; label proposed uses as suggestions. If no course is supplied do not infer lack of skill or invent a course. Never claim the proposed topic is an existing lab project or that access to data/equipment is available. Answer in Korean.'}
outputs=[];t=time.time()
for case in cases:
 for method in ['A','B']:
  messages=[{'role':'system','content':'You are an academic research exploration advisor. Avoid inventing facts.'},{'role':'user','content':rules[method]+'\nInput data:\n'+json.dumps(case,ensure_ascii=False)}]
  prompt=tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
  inputs=tokenizer(prompt,return_tensors='pt').to(model.device)
  with torch.inference_mode(): generated=model.generate(**inputs,max_new_tokens=250,do_sample=False,pad_token_id=tokenizer.eos_token_id)
  text=tokenizer.decode(generated[0,inputs['input_ids'].shape[1]:],skip_special_tokens=True)
  outputs.append({'case_id':case['id'],'method':method,'text':text,'tokens':int(generated.shape[1]-inputs['input_ids'].shape[1])})
  print(case['id'],method,flush=True)
json.dump({'model':model_id,'gpu':torch.cuda.get_device_name(0),'load_s':load,'generation_s':time.time()-t,'outputs':outputs},open('/kaggle/working/generations.json','w'),ensure_ascii=False,indent=2)
